In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Cell 1 — Check GPU & install missing library

In [ ]:
import tensorflow as tf
print("GPU available:", tf.config.list_physical_devices('GPU'))
print("TF version:", tf.__version__)

!pip install -q seaborn

Cell 2 — Imports & config

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow.keras.applications.efficientnet as efficientnet_preprocess

# ── CONFIG
IMG_SIZE    = 224
BATCH_SIZE  = 16
NUM_CLASSES = 5
SAVE_DIR    = "/kaggle/working"

CLASS_NAMES = {0: "No DR", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Proliferative"}

print("Setup complete ✓")

Cell 3 — Load & verify dataset


In [ ]:
TRAIN_CSV = "/kaggle/input/competitions/aptos2019-blindness-detection/train.csv"
TRAIN_DIR = "/kaggle/input/competitions/aptos2019-blindness-detection/train_images/"

df = pd.read_csv(TRAIN_CSV)
print("Columns:", df.columns.tolist())
print("Total images:", len(df))
print("\nClass distribution:")
print(df["diagnosis"].value_counts().sort_index())

df["image_path"] = df["id_code"].apply(lambda x: os.path.join(TRAIN_DIR, x + ".png"))
df["diagnosis"]  = df["diagnosis"].astype(str)

# Verify paths exist
missing = df["image_path"].apply(lambda p: not os.path.exists(p)).sum()
print(f"\nMissing images: {missing}")

Cell 4 — Train/val split


In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["diagnosis"],
    random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")

# Class weights to handle imbalance
classes = sorted(train_df["diagnosis"].unique())
weights = compute_class_weight("balanced", classes=np.array(classes), y=train_df["diagnosis"])
class_weights = {int(k): v for k, v in zip(classes, weights)}
print("Class weights:", class_weights)

Cell 5 — Sanity check (see one preprocessed image)

In [ ]:
def crop_image_from_gray(img, tol=7):
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    if img[:,:,0][np.ix_(mask.any(1), mask.any(0))].shape[0] == 0:
        return img
    return np.stack([img[:,:,c][np.ix_(mask.any(1), mask.any(0))] for c in range(3)], axis=-1)

def preprocess_image(path, sigmaX=10):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = crop_image_from_gray(img)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0,0), sigmaX), -4, 128)
    return np.clip(img, 0, 255).astype(np.uint8)

# Show original vs preprocessed
sample_path  = train_df["image_path"].iloc[0]
sample_label = train_df["diagnosis"].iloc[0]
original     = cv2.resize(cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB), (IMG_SIZE, IMG_SIZE))
processed    = preprocess_image(sample_path)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(original);  axes[0].set_title(f"Original (Class {sample_label})")
axes[1].imshow(processed); axes[1].set_title("After preprocessing")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print("Preprocessing looks good ✓")

Cell 6 — Build data generators

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=efficientnet_preprocess.preprocess_input,
    rotation_range=360,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.1,
    brightness_range=[0.8, 1.2],
    fill_mode="constant",
    cval=0
)
val_datagen = ImageDataGenerator(
    preprocessing_function=efficientnet_preprocess.preprocess_input
)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df, x_col="image_path", y_col="diagnosis",
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True, seed=42
)
val_gen = val_datagen.flow_from_dataframe(
    dataframe=val_df, x_col="image_path", y_col="diagnosis",
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

print("Class indices:", train_gen.class_indices)

Cell 7 — Build EfficientNetB0 model

In [ ]:
def build_efficientnet():
    inp  = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    base = EfficientNetB0(include_top=False, weights="imagenet", input_tensor=inp)
    base.trainable = False   # freeze backbone for Phase 1

    x   = layers.GlobalAveragePooling2D()(base.output)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(256, activation="relu")(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs=inp, outputs=out)
    return model, base

model, base_model = build_efficientnet()
model.summary()

Cell 8 — Phase 1: Train head only (frozen backbone)

In [ ]:
cb_phase1 = [
    callbacks.ModelCheckpoint("/kaggle/working/eff_phase1_best.keras",
                               monitor="val_accuracy", save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor="val_loss", patience=5,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                patience=3, min_lr=1e-7, verbose=1)
]

model.compile(optimizer=optimizers.Adam(1e-3),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

print("Phase 1: Training head (backbone frozen)...")
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weights,
    callbacks=cb_phase1,
    verbose=1
)

Cell 9 — Phase 2: Fine-tune top layers

In [ ]:
# Unfreeze top 30 layers of the backbone
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

cb_phase2 = [
    callbacks.ModelCheckpoint("/kaggle/working/eff_phase2_best.keras",
                               monitor="val_accuracy", save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor="val_loss", patience=5,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                patience=3, min_lr=1e-7, verbose=1)
]

model.compile(optimizer=optimizers.Adam(1e-5),   # much lower LR for fine-tuning
              loss="categorical_crossentropy",
              metrics=["accuracy"])

print("Phase 2: Fine-tuning top 30 layers...")
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight=class_weights,
    callbacks=cb_phase2,
    verbose=1
)

# Save final model
model.save("/kaggle/working/efficientnet_final.keras")
print("Model saved ✓")

Cell 10 — Plot training curves

In [ ]:
acc      = history1.history["accuracy"]      + history2.history["accuracy"]
val_acc  = history1.history["val_accuracy"]  + history2.history["val_accuracy"]
loss     = history1.history["loss"]          + history2.history["loss"]
val_loss = history1.history["val_loss"]      + history2.history["val_loss"]
boundary = len(history1.history["accuracy"])
epochs   = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, m, vm, title in [(ax1, acc, val_acc, "Accuracy"), (ax2, loss, val_loss, "Loss")]:
    ax.plot(epochs, m,  "b-", label=f"Train {title}")
    ax.plot(epochs, vm, "r-", label=f"Val {title}")
    ax.axvline(x=boundary, color="gray", linestyle="--", label="Fine-tune start")
    ax.set_title(f"EfficientNetB0 — {title}")
    ax.set_xlabel("Epoch"); ax.legend()
plt.tight_layout(); plt.savefig("/kaggle/working/training_curves.png", dpi=150); plt.show()

Cell 11 — Evaluate (accuracy, F1, confusion matrix)

In [ ]:
val_gen.reset()
y_prob = model.predict(val_gen, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = val_gen.classes

print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_true, y_pred, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=list(CLASS_NAMES.values())))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES.values(), yticklabels=CLASS_NAMES.values())
plt.title("Confusion Matrix — EfficientNetB0")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout(); plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150); plt.show()

Cell 12 — Grad-CAM visualization

In [ ]:
def get_gradcam(model, img_batch, class_idx):
    last_conv = next(l.name for l in reversed(model.layers) if len(l.output.shape) == 4)
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch)
        score = preds[:, class_idx]
    grads   = tape.gradient(score, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0,1,2))
    heatmap = tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def show_gradcam(image_path, true_label=None):
    img_bgr  = cv2.imread(image_path)
    img_rgb  = cv2.resize(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), (IMG_SIZE, IMG_SIZE))
    inp      = efficientnet_preprocess.preprocess_input(img_rgb.astype("float32"))
    batch    = np.expand_dims(inp, 0)
    preds    = model.predict(batch, verbose=0)[0]
    cls      = int(np.argmax(preds))
    conf     = preds[cls]
    heatmap  = get_gradcam(model, batch, cls)
    hm_resized = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    hm_colored = cv2.cvtColor(cv2.applyColorMap(np.uint8(255*hm_resized), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    overlay    = cv2.addWeighted(img_rgb, 0.6, hm_colored, 0.4, 0)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"Predicted: {CLASS_NAMES[cls]} ({conf:.1%})"
                 + (f"  |  True: {CLASS_NAMES[int(true_label)]}" if true_label is not None else ""),
                 fontsize=13)
    axes[0].imshow(img_rgb);          axes[0].set_title("Original")
    axes[1].imshow(heatmap, cmap="jet"); axes[1].set_title("Grad-CAM")
    axes[2].imshow(overlay);          axes[2].set_title("Overlay")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()

# Run on 5 random validation images
for i in range(5):
    row = val_df.iloc[i]
    show_gradcam(row["image_path"], true_label=row["diagnosis"])

Cell 13 — Download your model